# Phần 1 — Câu hỏi Trắc nghiệm (MCQ)

Mỗi cell tính toán trực tiếp đáp án cho từng câu hỏi từ dữ liệu.

| Câu | Nội dung | Đáp án |
|-----|----------|--------|
| Q1  | Trung vị inter-order gap (khách có >1 đơn) | ? |
| Q2  | Segment có gross margin trung bình cao nhất | ? |
| Q3  | Lý do trả hàng phổ biến nhất (Streetwear)  | ? |
| Q4  | Traffic source có bounce_rate thấp nhất    | ? |
| Q5  | % order_items có promo_id không null       | ? |
| Q6  | Age group có avg orders/customer cao nhất  | ? |
| Q7  | Region tạo doanh thu cao nhất              | ? |
| Q8  | Payment method phổ biến nhất (cancelled)   | ? |
| Q9  | Size có tỷ lệ trả hàng cao nhất            | ? |
| Q10 | Installment plan có avg payment_value cao nhất | ? |

In [1]:
import pandas as pd
import numpy as np

DATA = '../data/'

orders      = pd.read_csv(DATA + 'orders.csv',      parse_dates=['order_date'])
order_items = pd.read_csv(DATA + 'order_items.csv')
products    = pd.read_csv(DATA + 'products.csv')
customers   = pd.read_csv(DATA + 'customers.csv',   parse_dates=['signup_date'])
returns     = pd.read_csv(DATA + 'returns.csv',     parse_dates=['return_date'])
web_traffic = pd.read_csv(DATA + 'web_traffic.csv', parse_dates=['date'])
geography   = pd.read_csv(DATA + 'geography.csv')
payments    = pd.read_csv(DATA + 'payments.csv')
sales       = pd.read_csv(DATA + 'sales.csv',       parse_dates=['Date'])

print('Data loaded.')
print(f'orders: {orders.shape}, order_items: {order_items.shape}, products: {products.shape}')

C:\Users\hp\AppData\Local\Temp\ipykernel_17712\2283417741.py:7: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(DATA + 'order_items.csv')


Data loaded.
orders: (646945, 8), order_items: (714669, 7), products: (2412, 8)


## Q1 — Trung vị inter-order gap (khách có > 1 đơn)

> Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp?

In [2]:
# Sort by customer + date, compute diff between consecutive orders per customer
orders_sorted = orders.sort_values(['customer_id', 'order_date'])
orders_sorted['prev_date'] = orders_sorted.groupby('customer_id')['order_date'].shift(1)
orders_sorted['gap_days'] = (orders_sorted['order_date'] - orders_sorted['prev_date']).dt.days

# Only customers with >1 order
multi_order_customers = orders_sorted['customer_id'].value_counts()
multi_order_customers = multi_order_customers[multi_order_customers > 1].index

gaps = orders_sorted[
    (orders_sorted['customer_id'].isin(multi_order_customers)) &
    (orders_sorted['gap_days'].notna())
]['gap_days']

median_gap = gaps.median()
print(f'Median inter-order gap: {median_gap:.1f} days')
print()
print('Distribution:')
print(gaps.describe())

# Answer choices: 30 / 90 / 180 / 365
print(f'\n=> Q1 Answer: ', end='')
for threshold, label in [(45,'A) 30 ngày'), (135,'B) 90 ngày'), (270,'C) 180 ngày'), (999,'D) 365 ngày')]:
    if median_gap < threshold:
        print(label)
        break

Median inter-order gap: 144.0 days

Distribution:
count    556699.000000
mean        285.592509
std         389.691558
min           0.000000
25%          46.000000
50%         144.000000
75%         357.000000
max        3785.000000
Name: gap_days, dtype: float64

=> Q1 Answer: C) 180 ngày


## Q2 — Segment có gross margin trung bình cao nhất

> (price − cogs) / price theo từng segment trong products.csv

In [3]:
products['gross_margin'] = (products['price'] - products['cogs']) / products['price']
segment_margin = products.groupby('segment')['gross_margin'].mean().sort_values(ascending=False)
print('Gross margin by segment:')
print(segment_margin.round(4))
print(f'\n=> Q2 Answer: {segment_margin.idxmax()} (margin = {segment_margin.max():.4f})')

Gross margin by segment:
segment
Standard       0.3134
Premium        0.2854
All-weather    0.2842
Activewear     0.2656
Performance    0.2636
Balanced       0.2580
Trendy         0.2408
Everyday       0.2363
Name: gross_margin, dtype: float64

=> Q2 Answer: Standard (margin = 0.3134)


## Q3 — Lý do trả hàng phổ biến nhất trong danh mục Streetwear

In [4]:
streetwear_products = products[products['category'] == 'Streetwear'][['product_id']]
returns_streetwear = returns.merge(streetwear_products, on='product_id')

reason_counts = returns_streetwear['return_reason'].value_counts()
print('Return reasons for Streetwear:')
print(reason_counts)
print(f'\n=> Q3 Answer: {reason_counts.idxmax()} ({reason_counts.max()} returns)')

Return reasons for Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

=> Q3 Answer: wrong_size (7626 returns)


## Q4 — Traffic source có bounce_rate trung bình thấp nhất

In [5]:
bounce_by_source = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print('Average bounce rate by source:')
print(bounce_by_source.round(4))
print(f'\n=> Q4 Answer: {bounce_by_source.idxmin()} (bounce_rate = {bounce_by_source.min():.4f})')

Average bounce rate by source:
traffic_source
email_campaign    0.0045
social_media      0.0045
paid_search       0.0045
referral          0.0045
organic_search    0.0045
direct            0.0045
Name: bounce_rate, dtype: float64

=> Q4 Answer: email_campaign (bounce_rate = 0.0045)


## Q5 — % order_items có promo_id không null

In [6]:
total_items = len(order_items)
items_with_promo = order_items['promo_id'].notna().sum()
pct = items_with_promo / total_items * 100

print(f'Total order items  : {total_items:,}')
print(f'Items with promo   : {items_with_promo:,}')
print(f'Percentage         : {pct:.2f}%')
print(f'\n=> Q5 Answer: ', end='')
for bound, label in [(18,'A) 12%'), (32,'B) 25%'), (46,'C) 39%'), (100,'D) 54%')]:
    if pct < bound:
        print(label)
        break

Total order items  : 714,669
Items with promo   : 276,316
Percentage         : 38.66%

=> Q5 Answer: C) 39%


## Q6 — Age group có avg orders/customer cao nhất

In [7]:
# Count orders per customer
orders_per_customer = orders.groupby('customer_id').size().reset_index(name='order_count')

# Merge with customers (non-null age_group only)
customers_age = customers[customers['age_group'].notna()][['customer_id', 'age_group']]
merged = customers_age.merge(orders_per_customer, on='customer_id', how='left')
merged['order_count'] = merged['order_count'].fillna(0)

avg_orders_by_age = merged.groupby('age_group')['order_count'].mean().sort_values(ascending=False)
print('Average orders per customer by age group:')
print(avg_orders_by_age.round(3))
print(f'\n=> Q6 Answer: {avg_orders_by_age.idxmax()} ({avg_orders_by_age.max():.3f} avg orders)')

Average orders per customer by age group:
age_group
55+      5.407
45-54    5.357
35-44    5.337
25-34    5.245
18-24    5.227
Name: order_count, dtype: float64

=> Q6 Answer: 55+ (5.407 avg orders)


## Q7 — Region tạo doanh thu cao nhất trong sales_train.csv

> Lưu ý: sales.csv là aggregated (Revenue tổng công ty), không có cột region. 
> Phải join qua orders → geography để phân bổ doanh thu theo region.

In [8]:
# Join orders with geography to get region per order
orders_geo = orders.merge(geography[['zip', 'region']], on='zip', how='left')

# Join with order_items to get revenue per order (quantity * unit_price - discount)
order_revenue = order_items.copy()
order_revenue['line_revenue'] = order_revenue['quantity'] * order_revenue['unit_price'] - order_revenue['discount_amount']
order_revenue_total = order_revenue.groupby('order_id')['line_revenue'].sum().reset_index()

orders_geo_rev = orders_geo.merge(order_revenue_total, on='order_id', how='left')

region_revenue = orders_geo_rev.groupby('region')['line_revenue'].sum().sort_values(ascending=False)
print('Total revenue by region (from transaction data):')
print(region_revenue.apply(lambda x: f'{x:,.0f}'))
print(f'\n=> Q7 Answer: {region_revenue.idxmax()}')

Total revenue by region (from transaction data):
region
East       7,291,150,819
Central    4,719,491,268
West       3,670,227,178
Name: line_revenue, dtype: str

=> Q7 Answer: East


## Q8 — Payment method phổ biến nhất trong đơn cancelled

In [9]:
cancelled_orders = orders[orders['order_status'] == 'cancelled']
payment_method_counts = cancelled_orders['payment_method'].value_counts()
print('Payment methods for cancelled orders:')
print(payment_method_counts)
print(f'\n=> Q8 Answer: {payment_method_counts.idxmax()} ({payment_method_counts.max():,} orders)')

Payment methods for cancelled orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

=> Q8 Answer: credit_card (28,452 orders)


## Q9 — Size có tỷ lệ trả hàng cao nhất

> return records / order_item records, join với products theo product_id

In [10]:
# Count order_items per size
oi_with_size = order_items.merge(products[['product_id', 'size']], on='product_id', how='left')
oi_count_by_size = oi_with_size.groupby('size').size().rename('total_items')

# Count returns per size
ret_with_size = returns.merge(products[['product_id', 'size']], on='product_id', how='left')
ret_count_by_size = ret_with_size.groupby('size').size().rename('total_returns')

# Only S, M, L, XL
sizes_df = pd.DataFrame({'total_items': oi_count_by_size, 'total_returns': ret_count_by_size})
sizes_df = sizes_df.loc[['S', 'M', 'L', 'XL']]
sizes_df['return_rate'] = sizes_df['total_returns'] / sizes_df['total_items']

print('Return rate by size:')
print(sizes_df.round(4))
print(f'\n=> Q9 Answer: {sizes_df["return_rate"].idxmax()} (rate = {sizes_df["return_rate"].max():.4f})')

Return rate by size:
      total_items  total_returns  return_rate
size                                         
S          172042           9723       0.0565
M          176428           9820       0.0557
L          173174           9741       0.0562
XL         193025          10655       0.0552

=> Q9 Answer: S (rate = 0.0565)


## Q10 — Installment plan có avg payment_value cao nhất

In [11]:
avg_payment_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print('Average payment_value by installments:')
print(avg_payment_by_installment.round(2))
print(f'\n=> Q10 Answer: {avg_payment_by_installment.idxmax()} kỳ ({avg_payment_by_installment.max():,.2f})')

Average payment_value by installments:
installments
6     24446.65
3     24399.64
12    24245.77
1     24113.27
2       708.47
Name: payment_value, dtype: float64

=> Q10 Answer: 6 kỳ (24,446.65)
